In [ ]:
import numpy as np
import scanpy as sc
import sys
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import rapids_singlecell as rps
import anndata as ad
sys.path.append('/home/kstasinos/mydata/ScanpyPlus') # Replace with the actual path
import Scanpyplus
import celltypist
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import decoupler as dc
sc.set_figure_params(dpi=300, dpi_save=100)
sc.set_figure_params(dpi=300, dpi_save=100)

In [ ]:
adata_sc= sc.read_h5ad('/mnt/dev1/kstasinos/cxg_kidney/HCKA_sc.h5ad')

In [ ]:
adata_sn= sc.read_h5ad('/mnt/dev1/kstasinos/cxg_kidney/HCKA_sn.h5ad')

In [ ]:
mt_genes = adata_sc.var_names.str.startswith('MT-')
rp_genes = adata_sc.var_names.str.startswith('RPS') | adata_sc.var_names.str.startswith('RPL')

# 2. Create a mask of genes to KEEP (Not MT AND Not RP)
genes_to_keep = ~(mt_genes | rp_genes)

# 3. Create a temporary "clean" view for training
# This doesn't duplicate memory, just creates a view
adata_sc = adata_sc[:, genes_to_keep].copy()
adata_sn = adata_sn[:, genes_to_keep].copy()

In [ ]:
adata = ad.concat([adata_sc, adata_sn], label="modality", keys=["sc", "sn"])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# 1. Prepare the data (again, excluding MT/Ribo)
X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
y = adata.obs['suspension_type'].values

# 2. Train a Random Forest Classifier
# n_jobs=-1 uses all your CPU cores to make it fast
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# 3. Extract the most important genes
importance_df = pd.DataFrame({
    'Gene': adata.var_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

In [ ]:
#The top 100 genes here are your "Core Modality Signature"
core_signature = importance_df.head(100)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

# Plot the importance scores of the top 100 genes in order
sns.lineplot(
    x=range(0, len(importance_df)), 
    y=importance_df['Importance'], 
)

plt.title("Random Forest Importance Elbow Plot", fontsize=14)
plt.xlabel("Gene Rank", fontsize=12)
plt.ylabel("Importance Score", fontsize=12)

# This helps you visually spot where the 'cliff' ends and the noise begins
plt.axhline(y=0.002, color='red', linestyle='--', label='Potential Cutoff') 

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

sns.lineplot(
    x=range(1, len(importance_df) + 1), 
    y=importance_df['Importance'], 
    linewidth=2
)

plt.title("Zoomed-In Importance Elbow Plot", fontsize=14)
plt.xlabel("Gene Rank", fontsize=12)
plt.ylabel("Importance Score", fontsize=12)

# === THE FIX: Zoom in on the first 150 genes ===
plt.xlim(0, 600) 

# Add a grid to make reading the exact rank easier
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
df_detection = pd.DataFrame({
    'Gene': adata.var_names,
    'sc_det_rate': np.asarray((adata_sc.X > 0).mean(axis=0)).flatten(),
    'sn_det_rate': np.asarray((adata_sn.X > 0).mean(axis=0)).flatten()
})

# Calculate the delta
df_detection['delta_detection'] = df_detection['sc_det_rate'] - df_detection['sn_det_rate']

clean_importance_df = importance_df[['Gene', 'Importance']].copy()

# 2. Merge safely with your new, accurate df_detection
final_df = clean_importance_df.merge(df_detection, on='Gene')

# 3. Assign the modality tag based on the clean delta
final_df['Enriched_In'] = final_df['delta_detection'].apply(
    lambda x: 'Single-Cell' if x > 0 else 'Single-Nucleus'
)



In [ ]:
# 4. View the corrected Top 30
final_signature = final_df.head(200)
display(final_signature)

In [ ]:
plt.figure(figsize=(15, 8))

# Plot the top 30 genes horizontally
sns.barplot(
    data=final_signature, # Your top 30 dataframe from earlier
    x='Gene', 
    y='Importance',
    hue='Enriched_In',
    palette={'Single-Cell': 'tab:purple', 'Single-Nucleus': 'tab:green'},
    dodge=False # Keeps the bars centered
)

plt.title("Top 30 Technology-Specific Genes", fontsize=14)
plt.xlabel("")
plt.ylabel("Predictive importance for sinlge cell or nucleus", fontsize=12)
plt.yticks(fontsize=10)
plt.xticks(fontsize=4, rotation=90)
plt.legend(title="Enriched In", loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
final_signature.to_excel('technology_gene.xlsx')

In [ ]:
rename_map = {
    # --- PROXIMAL TUBULE & PODOCYTES ---
    "PEC": "Proximal_Tubule", 'POD': "Proximal_Tubule",
    'POD_ECM':"Proximal_Tubule",
    'dPOD':"Proximal_Tubule",
    "PTS1": "Proximal_Tubule", "PTS2": "Proximal_Tubule", "PTS1/S2": "Proximal_Tubule", "PTS3": "Proximal_Tubule",
    "PTS2/S3": "Proximal_Tubule", "aPTS1/S2": "Proximal_Tubule", "aPTS3": "Proximal_Tubule", "aPT": "Proximal_Tubule",
    "cycPT": "Proximal_Tubule", "dPT": "Proximal_Tubule", "dPTS1/S2": "Proximal_Tubule",

    # --- LOOP OF HENLE & DISTAL ---
    "ATL": "Loop_of_Henle", "aATL_CCL2+": "Loop_of_Henle", "TAL": "Loop_of_Henle", 
    "TAL_medullary": "Loop_of_Henle", "aTAL": "Loop_of_Henle", "DTL1": "Loop_of_Henle", 
    "DTL2": "Loop_of_Henle", "DTL3": "Loop_of_Henle", "DTL1/2": "Loop_of_Henle", "dDTL1/2": "Loop_of_Henle",
    "DCT1": "Distal_Tubule", "DCT2": "Distal_Tubule", "aDCT": "Distal_Tubule", "MD": "Distal_Tubule",

    # --- COLLECTING SYSTEM ---
    "CNT-PC": "Collecting_System", "CNT-IC-A": "Collecting_System", "aCNT-PC": "Collecting_System", 'cycDCT':'Collecting_System',
    "CD-PC": "Collecting_System", "aCD-PC": "Collecting_System", "dCD-PC": "Collecting_System",
    "CD-IC-A": "Collecting_System", "CD-IC-B": "Collecting_System", "dCD-ICB": "Collecting_System", "aCD-IC-B": "Collecting_System",
    "PapE": "Collecting_System", "PapE1": "Collecting_System", "PapE2": "Collecting_System", "dPapE": "Collecting_System",

    # --- ENDOTHELIAL ---
    "EC-cap": "Endothelial", "EC-PTC": "Endothelial", "cycEC-PTC": "Endothelial", "dEC-PTC": "Endothelial",
    "EC-AEA": "Endothelial", "EC-AEA_SULF1+": "Endothelial", "EC-DVR": "Endothelial", "EC-AVR": "Endothelial",
    "EC-Glomerular_cap": "Endothelial", "EC-LYM": "Endothelial", "dEC-LYM": "Endothelial", "EC-angiogenic": "Endothelial",
    "dEC-venous": "Endothelial", "EC-venus": "Endothelial", "dEC-cap": "Endothelial",

    # --- STROMAL / MESENCHYMAL ---
    "VSMC": "Stromal", "VSMC_REN+": "Stromal", "MC": "Stromal", "MC_REN+": "Stromal",
    "Pericytes": "Stromal", "Myofibroblasts": "Stromal", "Perivascular_fibroblasts": "Stromal",
    "Fibroblasts_adventitial": "Stromal", "Fibroblasts_NEGR1+": "Stromal", "Fibroblasts_INHBA+": "Stromal",
    "Fibroblasts_cortex": "Stromal", "Norn_cells": "Stromal", "PAC": "Stromal", "Medullary_fibroblasts": "Stromal",
    "Adipocytes": "Stromal",

    # --- IMMUNE - MYELOID ---
    "Mono_classical_CD14+": "Myeloid", "Mono_classical_activated": "Myeloid", "Mono_classical_migratory": "Myeloid",
    "Mono_non-classical_CD16+": "Myeloid", "Mono_non-classical_activated": "Myeloid", "Mono-Mφ": "Myeloid",
    "Monocytes": "Myeloid", "DC1": "Myeloid", "DC2": "Myeloid", "pDCs": "Myeloid", "Mφ_Resident": "Myeloid",
    "Mφ_Resident_LYVE1+": "Myeloid", "Mφ_SPP1+": "Myeloid", "Mφ_TREM2+": "Myeloid", "Neutrophils": "Myeloid",
    "Neutrophils_activated": "Myeloid", "Mast cells": "Myeloid", "Mast cells_activated": "Myeloid",

    # --- IMMUNE - LYMPHOID ---
    "CD4_memory_resident": "Lymphoid", "CD4_central_memory": "Lymphoid", "CD4_activated_helper": "Lymphoid",
    "Tregs": "Lymphoid", "CD4_helper": "Lymphoid", "CD8_memory_resident": "Lymphoid", "CD8_effector_cytoxic": "Lymphoid",
    "CD8_activated_resident": "Lymphoid", "CD8_activated_cytotoxic": "Lymphoid", "CD8_activated_exhausted": "Lymphoid",
    "CD8_cycling": "Lymphoid", "CD8_effector": "Lymphoid", "gdT cells": "Lymphoid", "NKT": "Lymphoid",
    "NK_CD56bright_CD16+": "Lymphoid", "NK_CD56bright_CD16-": "Lymphoid", "NK_CD56dim_CD16+": "Lymphoid",
    "ILC3": "Lymphoid", "B cells": "Lymphoid", "Bcells_naive": "Lymphoid", "Bcells_memory": "Lymphoid",
    "Bcells_activated": "Lymphoid", "Plasma cells": "Lymphoid",

    # --- OTHER ---
    "Neuron": "Other", "Schwann cells": "Other", "Neuron_peripheral": "Other", "Ciliated": "Other",
    "RBC": "Other", "Reticulocytes": "Other", "Platelets": "Other"
}

adata.obs["celltype_l2"] = (
    adata.obs["fine_annotation"]
    .astype(str)          # break categorical
    .replace(rename_map) # merge labels
    .astype("category")  # recast
)

In [ ]:
hex_dict = {
    'EC-cap': '#000080',
    'PTS2/S3': '#0000cd',
    'Neutrophils_activated': '#c600cd',
    'PTS2': '#7fffd4',
    'Neutrophils': '#0000cd',
    'PTS1/S2': '#ff7faa',
    'dEC-PTC': '#483d8b',
    'PTS3': '#c29c36',
    'POD': '#bc8f8f',
    'PT_Zhang': '#006400',
    'Platelets': '#008000',
    'Bcells_naive': '#228b22',
    'Neuron': '#2e8b57',
    'CD4_memory_resident': '#3cb371',
    'PTS1': '#32cd32',
    'dEC-cap': '#6b8e23',
    'PapE2': '#7cfc00',
    'dPOD': '#ff0080',
    'dPT': '#adff2f',
    'RBC': '#9acd32',
    'dPapE': '#90ee90',
    'DCT2': '#800000',
    'aPT_Zhang': '#66cdaa',
    'POD_ECM': '#9acd32',
    'Tregs': '#00fa9a',
    'PAC': '#00ff7f',
    'ILC3': '#20b2aa',
    'dCD-PC': '#87ceeb',
    'Mast cells': '#00ced1',
    'dDTL1/2': '#40e0d0',
    'Mast cells_activated': '#cad100',
    'TAL': '#00897b',
    'NK_CD56bright_CD16+': '#00bfff',
    'aTAL': '#cfad4b',
    'CD4_activated_helper': '#87cefa',
    'PTS1/S2_medullary': '#87ceeb',
    'EC-Glomerular_cap': '#33a02c',
    'PapE1': '#1e90ff',
    'gdT cells': '#6495ed',
    'dCD-ICB': '#4169e1',
    'EC-DVR': '#0288d1',
    'aPTS3': '#ba55d3',
    'CD8_memory_resident': '#663399',
    'DTL2': '#8a2be2',
    'CD8_cycling': '#9932cc',
    'MD': '#f4511e',              
    'CD8_activated_exhausted': '#9370db',
    'DCT1': '#fdd835',
    'Mono-Mφ': '#e91e63',        
    'ATL': '#19e6bd',
    'cDC2': '#9932cc',
    'CNT-PC': '#aa6e28',
    'NKT': '#da70d6',
    'aCD-PC': '#ee82ee',
    'EC-angiogenic': '#b2df8a',
    'CD-IC-A': '#00acc1',
    'NK_CD56bright_CD16-': '#ff1493',
    'CD-PC': '#fabebe',
    'Bcells_memory': '#db7093',
    'DTL3': '#ff69b4',
    'Bcells_activated': '#ffb6c1',
    'Schwann cells': '#ffcc00',
    'Mφ_TREM2+': '#4682b4',
    'PEC': '#558b2f',
    'EC-venus': '#b22222',
    'CD-IC-B': '#d2f53c',
    'CD8_activated_resident': '#8b0000',
    'aATL_CCL2+': '#ff0000',
    'CD4_central_memory': '#cd5c5c',
    'CNT-IC-A': '#ffddbe',
    'NK_CD56dim_CD16+': '#fa8072',
    'aPTS1/S2': '#ff6347',
    'EC-AEA_SULF1+': '#283593',
    'CD-ICA_Zhang': '#ff4500',
    'EC-AEA': '#ddc3c3',
    'aPT': '#32cd32',
    'CD8_activated_cytotoxic': '#ff8c00',
    'DTL1': '#ffa500',
    'Mφ_Resident_LYVE1+': '#ff4500',
    'Plasma cells': '#c0c0c0',
    'pDCs': '#317fc7',
    'EC-PTC': '#e64a19',
    'Mφ_Resident': '#bc8f8f',
    'EC-LYM': '#6d4c41',
    'Mφ_SPP1+': '#8b4513',
    'MC': '#7cb342',
    'dEC-venous': '#f5deb3',
    'Mono_non-classical_activated': '#ffdab9',
    'Fibroblasts_NEGR1+': '#37474f',
    'cDC1': '#bdb76b',
    'Fibroblasts_adventitial': '#ff8f00',
    'Mono_non-classical_CD16+': '#f0e68c',
    'cycEC-PTC': '#eee8aa',
    'Mono_classical_CD14+': '#ffff54',
    'Pericytes': '#2e8b57',
    'CD8_effector_cytoxic': '#ffff00',
    'Mono_classical_activated': '#546e7a',  
    'Perivascular_fibroblasts': '#696969',
    'Mono_classical_migratory': '#808080',
    'Adipocytes': '#ffe119',
    'EC-AVR': '#ff7f00',
    'dEC-LYM': '#d3d3d3',
    'VSMC': '#00b894',
    'B cells': '#0082c8',
    'CD8_effector': '#ef6c00',      
    'CD4_helper': '#911eb4',
    'Fibroblasts_INHBA+': '#26c6da',
    'MC_REN+': '#ab47bc',            
    'Monocytes': '#6b8e23',
    'TAL_medullary': '#ffae42',
    'DTL1/2': '#ffebcd',
    'aCD-IC-B': '#7b68ee',
    'Fibroblasts_cortex': '#afeeee',
    'aCNT-PC': '#ff8c00',
    'aDCT': '#0277bd',
    'cycDCT': '#add8e6',
    'cycPT': '#ad1457',
    'Neuron_peripheral': '#2e8b57',
    'DC1': '#bdb76b',
    'DC2': '#e53935',                
    'Myofibroblasts': '#a52a2a',
    'Medullary_fibroblasts': '#9370db',
    'Ciliated': '#00b894',
    'dPTS1/S2': '#e84393',
    'VSMC_REN+': '#DE3163',
    'Reticulocytes': '#ffdead',
    'Macrophages_Resident_LYVE1+': '#ff4500',
    'Macrophages_TREM2+': '#4682b4',
    'Norn_cells': '#000000',
}
palette = [hex_dict[cat] for cat in adata.obs['fine_annotation'].cat.categories]
adata.uns['fine_annotation_colors'] = palette

In [ ]:
adata

In [ ]:
sc.tl.leiden(adata, resolution=1, method='igraph', neighbors_key='concat')

In [ ]:
adata

In [ ]:
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects

fig, ax = plt.subplots(1,1, figsize=(10,10), dpi=500)

sc.pl.embedding(
    adata,
    basis='X_umap',
    color='fine_annotation',
    frameon=False,
    legend_fontoutline=2,
    legend_loc=None,   # turn off automatic labels
    size=3,
    ax=ax,
    show=False
)

texts = []

for ct in adata.obs['fine_annotation'].cat.categories:
    idx = adata.obs['fine_annotation'] == ct
    x = np.median(adata.obsm['X_umap'][idx, 0])
    y = np.median(adata.obsm['X_umap'][idx, 1])
    
    txt = ax.text(x, y, ct, fontsize=7, color='black')
    
    # Add white outline
    txt.set_path_effects([
        PathEffects.Stroke(linewidth=1, foreground='white'),
        PathEffects.Normal()
    ])
    
    texts.append(txt)

adjust_text(
    texts, 
    force_text=(3, 3),    # Lower force can sometimes prevent "jitter"
    expand_text=(0.5, 1),   # Force labels to act like they are bigger
    expand_points=(0, 0), # Push labels away from the UMAP dots
    arrowprops=dict(
        arrowstyle='-', 
        color='grey', 
        lw=0.5, 
        alpha=0.5             
    ),
    lim=10                  # Increase iterations for complex UMAPs
)

plt.show()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(10,10), dpi=500)
sc.pl.embedding(
    adata,
    basis='X_umap',
    color='fine_annotation',
    frameon=False,
    legend_fontoutline=2,
    #legend_loc=None,   # turn off automatic labels
    size=3,
    ax=ax,
    show=True,
    
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming your AnnData object is named 'adata'

# Compute the crosstab to get the contribution of 'sampled_site_condition' for each 'fine_annotation' cluster
ct = pd.crosstab(adata.obs['leiden'], adata.obs['suspension_type'])

# Normalize the values by row (i.e., per cluster) to get the percentage contributions
ct_norm = ct.div(ct.sum(axis=1), axis=0)

ct_norm.plot(kind='barh', stacked=True, figsize=(8, 20), colormap='Dark2')
plt.grid(False)
plt.gca().invert_yaxis()
plt.title('Contribution of suspension type per Cluster')
plt.xlabel('Proportion of suspension type') # This is now the X-axis
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
palette.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Compute the crosstab to get the contribution of 'sampled_site_condition' for each 'fine_annotation' cluster
ct = pd.crosstab(adata.obs['leiden'], adata.obs['coarse_annotation'])

# Normalize the values by row (i.e., per cluster) to get the percentage contributions
ct_norm = ct.div(ct.sum(axis=1), axis=0)

ax = ct_norm.plot(kind='barh', stacked=True, figsize=(12, 20), colormap='tab20')
ax.invert_yaxis()  # puts cluster 0 at the top

plt.grid(False)
plt.title('Contribution of cell type per Cluster')
plt.xlabel('Cluster', fontsize=15)
plt.ylabel('Proportion of cell type', fontsize=15)
plt.xticks(rotation=90, fontsize=15)
plt.legend(title='suspension_type', bbox_to_anchor=(1.03, 1), loc='upper left')
plt.tight_layout()
#plt.savefig("suspension_type_contribution.png", dpi=300)
plt.show()

In [ ]:
final_signature['Gene'].head(8).to_list()

In [ ]:
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects
import matplotlib.pyplot as plt
import copy

fig, axs = plt.subplots(4, 2, figsize=(5, 20), dpi=200)

# 2. Prepare the colormap
my_cmap = copy.copy(plt.get_cmap('PuRd'))
my_cmap.set_under('lightgrey') # Optional: make zeros/low values distinct

# 3. Define the genes to match your 2x2 grid
genes = final_signature['Gene'].head(8).to_list()

# 4. Loop through axes and genes to plot them individually
for i, gene in enumerate(genes):
    # Flatten the 2x2 axes array to 1D for easy indexing
    ax = axs.flat[i] 
    
    sc.pl.embedding(
        adata,
        basis='X_umap',
        color=gene,
        cmap=my_cmap,
        vmin=0.0001,    
        frameon=False,
        size=10,
        ax=ax,          # Crucial: tell scanpy which subplot to use
        show=False
    )
    ax.set_title(gene) # Clean up titles if needed

plt.tight_layout()

In [ ]:
adata.write_h5ad('HCKA_concatpre.h5ad')

In [ ]:
ct_norm.to_csv('leiden_composition.csv')

## Analysis of single cell vs nucleus by logFC pseudobulking

In [ ]:
adata = sc.read_h5ad('HCKA_concat1.h5ad')

In [ ]:
adata.obs['fine_annotation'] = adata.obs['fine_annotation'].replace({'PapE': 'PapE1'}).astype('category')

In [ ]:
adata.obs.suspension_type

In [ ]:
adata.X = adata.layers['counts'].copy() 

pdata = dc.pp.pseudobulk(
    adata,
    sample_col="batch",
    groups_col=["suspension_type"],
    mode="sum",
)

In [ ]:
pdata

In [ ]:
dc.pl.filter_samples(
    adata=pdata,
    groupby=["suspension_type", 'batch'],
    min_cells=50,
    min_counts=300,
    figsize=(5, 8),
)

In [ ]:
dc.pp.filter_samples(pdata, min_cells=50, min_counts=300)

In [ ]:
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

# Build DESeq2 object
inference = DefaultInference(n_cpus=64)
dds = DeseqDataSet(
    adata=pdata,
    design_factors=['batch', 'suspension_type'],
    refit_cooks=True,
    inference=inference,
)

dds.deseq2()

stat_res = DeseqStats(dds, contrast=["suspension_type", "cell", "nucleus"])
stat_res.summary()

res_df = stat_res.results_df.dropna()

res_df = res_df.sort_values('padj')
print(res_df.head(10))

#res_df.to_csv(f'Pseudobulk_DE_{target_niche.replace(" ", "_")}.csv')

In [ ]:
stat_res.results_df

In [ ]:
stat_res.plot_MA()
plt.show()

In [ ]:
stat_res.lfc_shrink(coeff="suspension_type[T.nucleus]")

res_df = stat_res.results_df.dropna()

res_df['log2FoldChange'] = res_df['log2FoldChange'] * -1

In [ ]:
res_df

In [ ]:
res_df.to_excel('pydeseq_sn_vs_sc.xlsx')

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(
    res_df['baseMean'], 
    res_df['log2FoldChange'], 
    c=np.where(res_df['padj'] < 0.05, '#d62728', 'grey'), 
    s=3, 
    alpha=0.5
)
plt.axhline(0, color='black', linestyle='--')
plt.xscale('log')
plt.xlabel('Mean of Normalized Counts')
plt.ylabel('Shrunken Log2 Fold Change (Positive = Cell, Negative = Nuc)')
plt.title('MA Plot: Shrunk LFCs')
plt.show()

In [ ]:
res_df_reset = res_df.reset_index().rename(columns={'index': 'Gene'})
res_df_reset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. Load and Merge (Assuming data is prepped as before)
# ==========================================
excel_df = pd.read_excel('technology_gene.xlsx')
res_df= pd.read_excel('pydeseq_sn_vs_sc.xlsx')
tech_df = excel_df[['Gene', 'delta_detection']] 
res_df_reset = res_df.reset_index().rename(columns={'Unnamed: 0': 'Gene'})
res_df_reset['-log10(padj)'] = -np.log10(res_df_reset['padj'] + 1e-300)
merged_df = pd.merge(res_df_reset, tech_df, on='Gene', how='inner')

# ==========================================
# 2. Build the Base Plot
# ==========================================
plt.figure(figsize=(15, 20), dpi=500)

scatter = plt.scatter(
    merged_df['log2FoldChange'], 
    merged_df['delta_detection'], 
    c=merged_df['-log10(padj)'], 
    cmap='viridis',  
    s=40,           
    alpha=0.9,
    edgecolors='white',
    linewidth=0.5
)

cbar = plt.colorbar(scatter)
cbar.set_label('-Log10(Adjusted P-Value)', rotation=270, labelpad=20)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.9) 
plt.axhline(y=0, color='black', linestyle='--', alpha=0.9) 

# ==========================================
# 3. Define and Gather Labels
# ==========================================
texts = []

# A. The Strong Markers
top_cell = merged_df[(merged_df['log2FoldChange'] > 1) & (merged_df['delta_detection'] > 0.25)].nlargest(10, 'delta_detection')

# B. The Strong Markers (Nucleus) - FIXED to grab the extreme X-axis outliers!
# Grab 5 lowest by Y-axis (Penetrance) and 5 lowest by X-axis (Magnitude)
top_nuc_y = merged_df[(merged_df['log2FoldChange'] < -1) & (merged_df['delta_detection'] < -0.25)].nsmallest(5, 'delta_detection')
top_nuc_x = merged_df[(merged_df['log2FoldChange'] < -1) & (merged_df['delta_detection'] < -0.25)].nsmallest(5, 'log2FoldChange')
top_nuc = pd.concat([top_nuc_y, top_nuc_x]).drop_duplicates()

# C. "The Soup" -> Low Fold Change, High Delta
soup_outliers = merged_df[(merged_df['log2FoldChange'].abs() < 1.0) & (merged_df['delta_detection'].abs() > 0.25)].nlargest(5, 'delta_detection')

# D. "The Rare" -> Massive Fold Change, but widened the Delta net to 0.15
rare_outliers = merged_df[(merged_df['log2FoldChange'].abs() > 3.0) & (merged_df['delta_detection'].abs() < 0.25)].nlargest(5, 'log2FoldChange')

# Add Standard Markers (Black, Bold)
for _, row in pd.concat([top_cell, top_nuc]).iterrows():
    texts.append(plt.text(row['log2FoldChange'], row['delta_detection'], row['Gene'], 
                          fontsize=12, fontweight='bold', color='black'))

# Add "Soup" outliers (Red, Italic)
for _, row in soup_outliers.iterrows():
    texts.append(plt.text(row['log2FoldChange'], row['delta_detection'], f"{row['Gene']} (Nuclear)", 
                          fontsize=12, style='italic', color='#d62728')) # Muted red

# Add "Rare" outliers (Blue, Italic)
for _, row in rare_outliers.iterrows():
    texts.append(plt.text(row['log2FoldChange'], row['delta_detection'], f"{row['Gene']} (Rare)", 
                          fontsize=12, style='italic', color='#1f77b4')) # Muted blue

# ==========================================
# 4. Apply the Text Overlap
# ==========================================
# This one line pushes all labels apart and draws lines to their dots
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.7))

# ==========================================
# 5. Clean up formatting
# ==========================================
plt.title('Magnitude vs. Penetrance of Modality genes', fontsize=25, pad=15)
plt.xlabel('Log2 Fold Change (Positive = Cell, Negative = Nucleus)', fontsize=20)
plt.ylabel('Delta Detection Rate (Positive = Cell Bias, Negative = Nucleus Bias)', fontsize=20)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

In [ ]:
top_cell['Gene'].to_list()

In [ ]:
top_nuc['Gene'].to_list()

In [ ]:
Scanpyplus.CellorNuc(
    adata1, 
    sc_genes=None,
    sn_genes=None,
    cell_type_col='fine_annotation', 
    assay_col='modality',  
    sn_label='sn',
    clip_percentiles=(0.01, 0.99),
    sc_threshold_percentile=0.05,
    sn_threshold_percentile=0.95,
    copy=False)

In [ ]:
custom_palette = {
    'Typical Cell': '#1f77b4',
    'Nucleus-like Cell': '#17becf',
    'Typical Nucleus': '#ff7f0e',
    'Cell-like Nucleus': '#d62728',
    'Uncertain': '#d3d3d3' # Now actively used for the middle gap
}


df = adata.obs

# 2. Find the extreme 1st and 99th percentiles of this unified score
vmin, vmax = df['cell_vs_nucleus_score'].quantile([0.01, 0.99])

# 3. Clip the outliers. This mathematically "zooms in" and spreads the dense center out
df['score_clipped'] = df['cell_vs_nucleus_score'].clip(vmin, vmax)

# 4. Sort cell types by their median clipped score
cell_type_order = df.groupby(cell_type_col)['score_clipped'].median().sort_values(ascending=False).index.tolist()

# 1. Define logical masks for your three tiers of importance
mask_uncertain = df['classification'] == 'Uncertain'
mask_typical = df['classification'].isin(['Typical Cell', 'Typical Nucleus'])
mask_anomaly = df['classification'].isin(['Nucleus-like Cell', 'Cell-like Nucleus'])

# 2. Setup the figure
fig_height = max(8, len(cell_type_order) * 0.6)
fig, ax = plt.subplots(figsize=(12, fig_height), dpi=300)

# 3. Base Layer
sns.violinplot(
    data=df, x='score_clipped', y=cell_type_col, order=cell_type_order,
    color='lightgrey', inner=None, linewidth=0, alpha=0.2, ax=ax
)

# 4. Layer 1: Uncertain Cells (Gray)
# Lowest alpha (0.15) and lowest z-order so they stay in the deep background
if mask_uncertain.any():
    sns.stripplot(
        data=df[mask_uncertain], x='score_clipped', y=cell_type_col,
        hue='classification', order=cell_type_order, palette=custom_palette,
        dodge=False, jitter=0.35, size=1.0, alpha=0.2, zorder=1, ax=ax
    )

# 5. Layer 2: Typical Cells & Nuclei (Blue and Orange)
# Medium alpha (0.3) and normal size so they show the distribution without dominating
if mask_typical.any():
    sns.stripplot(
        data=df[mask_typical], x='score_clipped', y=cell_type_col,
        hue='classification', order=cell_type_order, palette=custom_palette,
        dodge=False, jitter=0.35, size=1.5, alpha=0.2, zorder=2, ax=ax
    )

# 6. Layer 3: The Anomalies (Red and Cyan)
# High alpha (0.9), higher z-order (drawn on top), and slightly larger size!
if mask_anomaly.any():
    sns.stripplot(
        data=df[mask_anomaly], x='score_clipped', y=cell_type_col,
        hue='classification', order=cell_type_order, palette=custom_palette,
        dodge=False, jitter=0.35, size=2.5, alpha=0.9, zorder=3, ax=ax
    )

# 7. Add the central zero line
ax.axvline(0, color='black', linewidth=1.5, zorder=4, alpha=0.8, linestyle='--')

# 8. Formatting
ax.set_xlabel('← More Nucleus-Like        |        More Cell-Like →')
ax.set_ylabel('Cell Type')
ax.set_title('Distribution of Cell vs Nucleus Profiles')

# 9. Clean up the legend
# Because we called stripplot 3 times, we will have duplicate legend entries.
# This dictionary trick automatically removes the duplicates.
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=3)

plt.tight_layout()
plt.savefig('sc_sn.png')
plt.show()

In [ ]:
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects

custom_palette = {
    'Typical Cell': '#1f77b4',
    'Nucleus-like Cell': '#17becf',
    'Typical Nucleus': '#ff7f0e',
    'Cell-like Nucleus': '#d62728',
    'Uncertain': '#d3d3d3' # Now actively used for the middle gap
}

fig, ax = plt.subplots(1,1, figsize=(5,5), dpi=500)

sc.pl.embedding(
    adata,
    basis='X_umap',
    color='modality_classification',
    frameon=False,
    legend_fontoutline=2,
    legend_loc=None,   # turn off automatic labels
    size=3,
    palette=custom_palette,
    ax=ax,
    show=False
)

texts = []

for ct in adata.obs['modality_classification'].cat.categories:
    idx = adata.obs['modality_classification'] == ct
    x = np.median(adata.obsm['X_umap'][idx, 0])
    y = np.median(adata.obsm['X_umap'][idx, 1])
    
    txt = ax.text(x, y, ct, fontsize=8, color='black')
    
    # Add white outline
    txt.set_path_effects([
        PathEffects.Stroke(linewidth=1, foreground='white'),
        PathEffects.Normal()
    ])
    
    texts.append(txt)

adjust_text(
    texts, 
    force_text=(3, 3),    # Lower force can sometimes prevent "jitter"
    expand_text=(0.5, 1),   # Force labels to act like they are bigger
    expand_points=(0, 0), # Push labels away from the UMAP dots
    arrowprops=dict(
        arrowstyle='-', 
        color='grey', 
        lw=0.5, 
        alpha=0.5             # Lighter arrows make the plot feel less cluttered
    ),
    lim=10000                  # Increase iterations for complex UMAPs
)

plt.show()

In [ ]:
adata.obs['modality_classification'] = df['classification']

In [ ]:
adata.obs['cell_modality'] = adata.obs['fine_annotation'].astype(str) + "_" + adata.obs['modality'].astype(str)

In [ ]:
adata.obs['cell_modality'] = adata.obs['cell_modality'].astype('category')

In [ ]:
crosscell = pd.crosstab(adata.obs['leiden'], adata.obs['cell_modality'])

In [ ]:
crosscell.to_csv('cell_modality_counts.csv')

In [ ]:
crosscell

In [ ]:
adata.write_h5ad('HCKA_concat1.h5ad')

In [ ]:
## function

import scanpy as sc
import pandas as pd
import numpy as np

sc_genes = ['EEF1A1', 'TPT1', 'FTH1', 'FTL', 'SERF2', 'ATP5F1E', 'GAPDH', 'UBA52', 'COX7C', 'MIF']
sn_genes = ['FTX', 'AGAP1', 'GMDS-DT', 'PARD3', 'WWOX', 'MAGI2', 'PKHD1', 'NHS', 'AC098829.1', 'DPP6']

def CellorNuc(
    adata, 
    sc_genes, 
    sn_genes, 
    cell_type_col='cell_type', 
    assay_col='assay', 
    sc_label='sc', 
    sn_label='sn',
    clip_percentiles=(0.01, 0.99),
    sc_threshold_percentile=0.05,
    sn_threshold_percentile=0.95,
    copy=False
):
    """
    Scores single cells and single nuclei based on distinct gene signatures and 
    classifies anomalous cross-modality profiles.
    
    Parameters
    ----------
    adata : AnnData
        The annotated data matrix of shape n_obs × n_vars.
    sc_genes : list
        List of genes enriched in single-cell assays (e.g., cytoplasmic markers).
    sn_genes : list
        List of genes enriched in single-nucleus assays (e.g., nuclear markers).
    cell_type_col : str, optional (default: 'cell_type')
        Column in adata.obs containing cell type annotations.
    assay_col : str, optional (default: 'assay')
        Column in adata.obs containing the modality assay labels.
    sc_label : str, optional (default: 'sc')
        The exact string in assay_col identifying single-cell droplets.
    sn_label : str, optional (default: 'sn')
        The exact string in assay_col identifying single-nucleus droplets.
    clip_percentiles : tuple, optional (default: (0.01, 0.99))
        Percentiles used to clip (Winsorize) the differential score to prevent outliers.
    sc_threshold_percentile : float, optional (default: 0.05)
        The lower-bound percentile for typical SC profiles. Droplets falling below this 
        but above the SN upper bound become 'Uncertain'.
    sn_threshold_percentile : float, optional (default: 0.95)
        The upper-bound percentile for typical SN profiles. Droplets exceeding this 
        but falling below the SC lower bound become 'Uncertain'.
    copy : bool, optional (default: False)
        If True, returns a copy of the AnnData object. Otherwise, modifies in place.
        
    Returns
    -------
    AnnData (if copy=True)
        Updates adata.obs with score columns and 'modality_classification'.
    """
    
    adata = adata.copy() if copy else adata
    
    # 1. Validate inputs
    if cell_type_col not in adata.obs.columns:
        raise ValueError(f"Column '{cell_type_col}' not found in adata.obs")
    if assay_col not in adata.obs.columns:
        raise ValueError(f"Column '{assay_col}' not found in adata.obs")
        
    # 2. Filter for genes actually present in the matrix
    sc_found = [g for g in sc_genes if g in adata.var_names]
    sn_found = [g for g in sn_genes if g in adata.var_names]
    
    if not sc_found or not sn_found:
        raise ValueError("One or both gene lists have zero overlap with adata.var_names")
        
    # 3. Score genes globally
    sc.tl.score_genes(adata, gene_list=sc_found, score_name='sc_score')
    sc.tl.score_genes(adata, gene_list=sn_found, score_name='sn_score')
    
    # 4. Calculate differential score and clip outliers
    adata.obs['cell_vs_nucleus_score'] = adata.obs['sc_score'] - adata.obs['sn_score']
    
    vmin, vmax = adata.obs['cell_vs_nucleus_score'].quantile(list(clip_percentiles))
    adata.obs['cell_vs_nucleus_score_clipped'] = adata.obs['cell_vs_nucleus_score'].clip(vmin, vmax)
    
    # 5. Initialize classification column
    adata.obs['modality_classification'] = 'Uncertain'
    
    # 6. Apply cell-type specific statistical boundaries
    for ct in adata.obs[cell_type_col].unique():
        ct_mask = adata.obs[cell_type_col] == ct
        sc_mask = ct_mask & (adata.obs[assay_col] == sc_label)
        sn_mask = ct_mask & (adata.obs[assay_col] == sn_label)
        
        # Determine SC lower bound using the parameterized percentile (min 0)
        if sc_mask.any():
            sc_lower = max(0.0, adata.obs.loc[sc_mask, 'cell_vs_nucleus_score'].quantile(sc_threshold_percentile))
        else:
            sc_lower = 0.0
            
        # Determine SN upper bound using the parameterized percentile (max 0)
        if sn_mask.any():
            sn_upper = min(0.0, adata.obs.loc[sn_mask, 'cell_vs_nucleus_score'].quantile(sn_threshold_percentile))
        else:
            sn_upper = 0.0
            
        # Classify Single-Cell droplets
        if sc_mask.any():
            adata.obs.loc[sc_mask & (adata.obs['cell_vs_nucleus_score'] >= sc_lower), 'modality_classification'] = 'Typical Cell (SC)'
            adata.obs.loc[sc_mask & (adata.obs['cell_vs_nucleus_score'] <= sn_upper), 'modality_classification'] = 'Nucleus-like Cell (SC)'
            
        # Classify Single-Nucleus droplets
        if sn_mask.any():
            adata.obs.loc[sn_mask & (adata.obs['cell_vs_nucleus_score'] <= sn_upper), 'modality_classification'] = 'Typical Nucleus (SN)'
            adata.obs.loc[sn_mask & (adata.obs['cell_vs_nucleus_score'] >= sc_lower), 'modality_classification'] = 'Cell-like Nucleus (SN)'
            
    # Convert to categorical for better memory efficiency and downstream plotting
    categories = ['Typical Cell', 'Nucleus-like Cell', 'Typical Nucleus', 'Cell-like Nucleus', 'Uncertain']
    adata.obs['modality_classification'] = pd.Categorical(adata.obs['modality_classification'], categories=categories)
    
    return adata if copy else None

In [ ]:
adata1 = adata[adata.obs.modality.isin(['sn'])].copy()

In [ ]:
custom_palette = {
    'Typical Cell': '#1f77b4',
    'Nucleus-like Cell': '#17becf',
    'Typical Nucleus': '#ff7f0e',
    'Cell-like Nucleus': '#d62728',
    'Uncertain': '#d3d3d3' # Now actively used for the middle gap
}


df = adata1.obs

# 2. Find the extreme 1st and 99th percentiles of this unified score
vmin, vmax = df['cell_vs_nucleus_score'].quantile([0.01, 0.99])

# 3. Clip the outliers. This mathematically "zooms in" and spreads the dense center out
df['score_clipped'] = df['cell_vs_nucleus_score'].clip(vmin, vmax)

# 4. Sort cell types by their median clipped score
cell_type_order = df.groupby('fine_annotation')['score_clipped'].median().sort_values(ascending=False).index.tolist()

# 1. Define logical masks for your three tiers of importance
mask_uncertain = df['modality_classification'] == 'Uncertain'
mask_typical = df['modality_classification'].isin(['Typical Cell (SC)', 'Typical Nucleus (SN)'])
mask_anomaly = df['modality_classification'].isin(['Nucleus-like Cell (SN)', 'Cell-like Nucleus (SC)'])

# 2. Setup the figure
fig_height = max(8, len(cell_type_order) * 0.6)
fig, ax = plt.subplots(figsize=(12, fig_height), dpi=300)

# 3. Base Layer
sns.violinplot(
    data=df, x='score_clipped', y='fine_annotation', order=cell_type_order,
    color='lightgrey', inner=None, linewidth=0, alpha=0.2, ax=ax
)

# 4. Layer 1: Uncertain Cells (Gray)
# Lowest alpha (0.15) and lowest z-order so they stay in the deep background
if mask_uncertain.any():
    sns.stripplot(
        data=df[mask_uncertain], x='score_clipped', y='fine_annotation',
        hue='modality_classification', order=cell_type_order, palette=custom_palette,
        dodge=False, jitter=0.35, size=1.0, alpha=0.2, zorder=1, ax=ax
    )

# 5. Layer 2: Typical Cells & Nuclei (Blue and Orange)
# Medium alpha (0.3) and normal size so they show the distribution without dominating
if mask_typical.any():
    sns.stripplot(
        data=df[mask_typical], x='score_clipped', y='fine_annotation',
        hue='modality_classification', order=cell_type_order, palette=custom_palette,
        dodge=False, jitter=0.35, size=1.5, alpha=0.2, zorder=2, ax=ax
    )

# 6. Layer 3: The Anomalies (Red and Cyan)
# High alpha (0.9), higher z-order (drawn on top), and slightly larger size!
if mask_anomaly.any():
    sns.stripplot(
        data=df[mask_anomaly], x='score_clipped', y=cell_type_col,
        hue='modality_classification', order=cell_type_order, palette=custom_palette,
        dodge=False, jitter=0.35, size=2.5, alpha=0.9, zorder=3, ax=ax
    )

# 7. Add the central zero line
ax.axvline(0, color='black', linewidth=1.5, zorder=4, alpha=0.8, linestyle='--')

# 8. Formatting
ax.set_xlabel('← More Nucleus-Like        |        More Cell-Like →')
ax.set_ylabel('Cell Type')
ax.set_title('Distribution of Cell vs Nucleus Profiles')

# 9. Clean up the legend
# Because we called stripplot 3 times, we will have duplicate legend entries.
# This dictionary trick automatically removes the duplicates.
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=3)

plt.tight_layout()
#plt.savefig('sc_sn.png')
plt.show()